# Assignment 2.6 - FLUX.1 (Diffusers)

Infer Flux qua Diffusers:
- **1.1 Text2Img** voi `FluxPipeline`
- **1.2 Img2Img / Image Editing** voi `FluxImg2ImgPipeline` (+ `FluxKontextPipeline`)
- ComfyUI workflow cho Flux

> Flux ~12B. Nen GPU >= 16GB + `enable_model_cpu_offload()`. `FLUX.1-dev` GATED (can login + accept license).
> Anh luu vao `output/`.

In [ ]:
import os, torch
from diffusers.utils import load_image

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32
print("device:", device, "| dtype:", dtype)

# Diffusers 0.38 dang ky custom-op voi annotation hoan (postponed); torch 2.4.x khong
# tu suy ra schema -> resolve type hints truoc khi import Flux/Qwen de tranh loi infer_schema.
import typing
import torch._library.infer_schema as _infer_schema_mod
_orig_infer_schema = _infer_schema_mod.infer_schema

def _patched_infer_schema(prototype_function, mutates_args):
    try:
        hints = typing.get_type_hints(
            prototype_function, globalns=getattr(prototype_function, "__globals__", {}))
    except Exception:
        hints = {}
    if hints:
        orig = dict(getattr(prototype_function, "__annotations__", {}))
        prototype_function.__annotations__ = {**orig, **hints}
        try:
            return _orig_infer_schema(prototype_function, mutates_args)
        finally:
            prototype_function.__annotations__ = orig
    return _orig_infer_schema(prototype_function, mutates_args)

_infer_schema_mod.infer_schema = _patched_infer_schema
try:
    import torch._custom_op.impl as _custom_impl
    _custom_impl.infer_schema = _patched_infer_schema
except Exception:
    pass
print("patched torch schema inference for diffusers import")

In [ ]:
# (1) Dang nhap HF MA KHONG ghi token vao notebook.
#     Thu tu lay token: Colab Secrets -> file .env -> bien moi truong -> nhap tay.
import pathlib
from huggingface_hub import login

def _load_dotenv():
    """Doc file .env (tim o cwd va cac thu muc cha) roi nap vao os.environ."""
    here = pathlib.Path.cwd()
    for d in [here, *here.parents]:
        f = d / ".env"
        if f.is_file():
            for line in f.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
            return str(f)
    return None

def get_hf_token():
    # a) Colab Secrets (icon chia khoa ben trai Colab -> them HF_TOKEN, bat Notebook access)
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            return tok
    except Exception:
        pass
    # b) file .env (sao chep .env.example -> .env va dien token that)
    _load_dotenv()
    tok = os.environ.get("HF_TOKEN", "")
    if tok and not tok.startswith("hf_your"):     # bo qua gia tri placeholder
        return tok
    # c) nhap tay - an tren man hinh, khong luu vao file
    from getpass import getpass
    return getpass("Nhap HF token (https://huggingface.co/settings/tokens): ")

login(token=get_hf_token())
print("HF login OK")
# Nho: vao trang model va bam "Agree" license (vd FLUX.1-dev) thi moi tai duoc.

# (2) Dung luong dia: Flux ~24GB, Qwen-Image ~40GB. Kiem tra cho trong:
import shutil
free_gb = shutil.disk_usage("/root").free / 1e9
print(f"Free disk: {free_gb:.1f} GB")
if free_gb < 30:
    print("!! Thieu dia. Don cache model cu hoac dung runtime GPU lon hon (A100).")

# Model dung chung:
#   FLUX.1-dev    : chat luong cao  (guidance 3.5, ~50 steps) - GATED
#   FLUX.1-schnell: nhanh, nhe hon  (guidance 0.0, ~4 steps)
FLUX_MODEL = "black-forest-labs/FLUX.1-dev"

## 1.1 Text2Img - `FluxPipeline`

In [ ]:
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained(FLUX_MODEL, torch_dtype=dtype)
pipe.enable_model_cpu_offload()          # tiet kiem VRAM (chay duoc ~16GB GPU, cham hon)

prompt = "a cinematic photo of a small cabin in a snowy forest at sunset, highly detailed"
image = pipe(
    prompt,
    height=1024, width=1024,
    guidance_scale=3.5,                  # FLUX.1-dev: 3.5  |  FLUX.1-schnell: 0.0
    num_inference_steps=50,              # FLUX.1-dev: ~50  |  FLUX.1-schnell: ~4
    max_sequence_length=512,             # FLUX.1-schnell: 256
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]

image.save(os.path.join(OUTPUT_DIR, "flux_t2i.png"))
print("saved output/flux_t2i.png")
image

## 1.2 Img2Img & Image Editing

- **Img2Img:** `FluxImg2ImgPipeline` - giu bo cuc anh goc, ve lai theo prompt (`strength`).
- **Editing theo lenh:** `FluxKontextPipeline` voi `FLUX.1-Kontext-dev`.

In [ ]:
from diffusers import FluxImg2ImgPipeline

pipe_i2i = FluxImg2ImgPipeline.from_pretrained(FLUX_MODEL, torch_dtype=dtype)
pipe_i2i.enable_model_cpu_offload()

init_image = load_image(
    "https://raw.githubusercontent.com/CompVis/stable-diffusion/main/assets/stable-samples/img2img/sketch-mountains-input.jpg"
).resize((1024, 1024))

image = pipe_i2i(
    prompt="a fantasy landscape, vivid autumn colors, highly detailed",
    image=init_image,
    strength=0.6,                  # 0 = giu nguyen anh goc, 1 = ve moi hoan toan
    guidance_scale=3.5,
    num_inference_steps=40,
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]

image.save(os.path.join(OUTPUT_DIR, "flux_i2i.png"))
print("saved output/flux_i2i.png")
image

In [ ]:
# (Tuy chon) Image Editing theo lenh voi FLUX.1-Kontext
# from diffusers import FluxKontextPipeline
# kontext = FluxKontextPipeline.from_pretrained("black-forest-labs/FLUX.1-Kontext-dev", torch_dtype=dtype)
# kontext.enable_model_cpu_offload()
# edited = kontext(image=init_image, prompt="make it winter with snow", guidance_scale=2.5).images[0]
# edited.save(os.path.join(OUTPUT_DIR, "flux_kontext_edit.png"))
print("Kontext: bo comment de chinh sua anh theo lenh (instruction editing).")

## ComfyUI - Workflow cho Flux

ComfyUI co san template: **Workflow -> Browse Templates -> Flux**. So do node:
```
UNETLoader(flux1-dev.safetensors)            --MODEL-->
DualCLIPLoader(clip_l + t5xxl, type=flux)    --CLIP--> CLIPTextEncode(prompt)
VAELoader(ae.safetensors)                    --VAE-->
[MODEL]+[CONDITIONING] -> SamplerCustomAdvanced (hoac KSampler)
EmptySD3LatentImage(1024x1024) -> sampler -> VAEDecode -> SaveImage
```
- **Img2Img:** thay `EmptySD3LatentImage` bang `LoadImage -> VAEEncode`, ha `denoise` (~0.5-0.7).
- **Editing:** dung node **FLUX.1-Kontext** (ReferenceLatent / Kontext).

**Tang chat luong:** Upscale (`UpscaleModelLoader` + `ImageUpscaleWithModel`, 4x-UltraSharp), hi-res 2 pass.
**Tang toc:** `FLUX.1-schnell` (4 buoc) hoac GGUF/fp8 (`UnetLoaderGGUF`); TeaCache; bat `--fast`/sage-attention.